In [1]:
import argparse

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn.functional as F
from DPAI_model import create_model

def visualize_attention(model_path, img_t, img_t1, img_t2, state_seq, save_path='attention_map.png'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. 加载模型
    model = create_model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. 前向传播
    with torch.no_grad():
        # preds, attn_map = model(img_t2, img_t1, img_t, state_seq, return_attn = True)
        preds, attn_map = model(img_t2, img_t1, img_t, state_seq)

    # 3. 处理注意力图 (假设形状为 [1, 1, H_feat, W_feat])
    attn_data = attn_map.cpu().squeeze().numpy()
    if len(attn_data.shape) == 3:
        attn_data = np.mean(attn_data, axis = 0)
    attn_data = (attn_data - attn_data.min()) / (attn_data.max() - attn_data.min() + 1e-8)
    # 归一化到 [0, 1]

    # 4. 还原原始图像 (从 Tensor 转回 numpy [H, W, C])
    # 假设 img_t 形状为 [1, 3, H, W]，且已经过正则化
    orig_img = img_t.cpu().squeeze().permute(1, 2, 0).numpy()

    # 如果你的图像之前做过 Normalize (mean, std)，这里需要反向操作才能正常显示
    # mean = np.array([0.485, 0.456, 0.406])
    # std = np.array([0.229, 0.224, 0.225])
    # orig_img = std * orig_img + mean
    orig_img = np.clip(orig_img, 0, 1) # 确保在 [0, 1] 范围内

    # 5. 绘图
    plt.figure(figsize=(12, 6))

    # 子图 1: 原图
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(orig_img)
    plt.axis('off')

    # 子图 2: 叠加图
    plt.subplot(1, 2, 2)
    plt.title("Attention Heatmap")
    plt.imshow(orig_img)

    # 使用 Matplotlib 的 colormap 覆盖注意力图
    # interpolation='bilinear' 相当于 OpenCV 的 resize，实现平滑上采样
    # alpha 控制透明度，cmap='jet' 是经典的红蓝热力图
    plt.imshow(attn_data, cmap='jet', alpha=0.5, interpolation='bilinear')

    plt.axis('off')

    # 保存与展示
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

if __name__ == "__main__":
    model_path = 'logs/3000_7000_HuberLoss.pth'
    output_csv = 'processed_data.csv'
    dataset = ProcessedDrivingDataset(csv_file=output_csv)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

    # 初始化模型、损失函数、优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for batch_idx, (images, state_seq, target) in enumerate(dataloader):
        # 数据移至设备
        img_t_minus_2, img_t_minus_1, img_t = [img.to(device) for img in images]
        state_seq = state_seq.to(device)
        target = target.to(device)
        visualize_attention(model_path, img_t_minus_2, img_t_minus_1, img_t, state_seq)
        break




ModuleNotFoundError: No module named 'cv2'